# CS781 Phase 2 - Legal NER on top of the Phase-1 LegalLM

One notebook, one shared pipeline, several adaptation methods. Every experiment uses the **same** data split, windows, labels, decoding and scorer; only the adaptation method changes.

| Method key | What is trained | Role in the report |
|---|---|---|
| `full` | every weight + NER head | full fine-tuning (also the KD teacher) |
| `lora` | low-rank additive updates `W0 + (a/r)BA` on all linear layers + head | parameter-efficient (additive) |
| `lorma` | low-rank multiplicative updates `(I + (a/r)BA)W0` (LoRMA+) on the same layers + head | **compulsory** LoRMA |
| `kd` | small student (every 2nd Phase-1 layer) trained on gold labels + teacher soft labels | **compulsory** knowledge distillation |
| `student` | same small student, gold labels only | KD control (shows what distillation adds) |
| `partial` | top-k layers + head | layer-wise fine-tuning (optional extra) |
| `adapter` | bottleneck adapters after every FFN + head | Pfeiffer adapters (optional extra) |

**Pipeline:** character spans -> BIO tags on tokens (whitespace-only tokens ignored) -> overlapping windows -> model -> probabilities averaged over windows (centre-weighted) -> **constrained Viterbi** (no `I-X` without `B-X`/`I-X` before it) -> character spans (token edges with whitespace trimmed) -> strict scoring identical to `evaluate_phase2.py`.

**How to run:** attach the Phase-2 data, `evaluate_phase2.py`, and your Phase-1 model folder (`config.json`, `pytorch_model.bin`, `tokenizer.json`) as inputs, then run all cells. The last section runs the experiments; each one writes `submission_<name>.zip` (fold3 predictions) and `metrics.json`.

## 1. Configuration

In [2]:
import os, sys, json, math, time, random, re, glob, zipfile, shutil, subprocess, contextlib, copy
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- paths (None = search /kaggle/input automatically)
DATA_DIR = None            # folder that contains Train/fold1-text.json (or the files directly)
PHASE1_MODEL_DIR = None    # LegalLM folder: config.json + pytorch_model.bin + tokenizer.json
EVALUATOR = None           # evaluate_phase2.py
OUT_ROOT = "/kaggle/working/lner"

# ---- split: validation documents taken from each training fold (document level, fixed seed)
SEED = 42
VAL_DOCS_PER_FOLD = 7      # 7 + 7 of the 70 annotated documents are held out for model selection
DEBUG_DOCS = 0             # >0: use only this many docs per fold (quick smoke test)

# ---- shared defaults for every experiment (override per run in run_experiment(..., key=value))
DEFAULTS = dict(
    bidirectional=True,    # remove the causal mask during fine-tuning (NER needs right context too)
    window=512, stride=384,
    epochs=20, patience=6, batch_size=8, eval_batch_size=32,
    lr=5e-5, head_lr=1e-3, weight_decay=0.01, warmup=0.1, grad_clip=1.0, dropout=0.1,
    loss="ce",             # "ce" | "weighted" (sqrt inverse-frequency class weights)
    o_weight=1.0,          # extra multiplier on the O class weight (e.g. 0.5 to favour recall)
    fp16=True,
    refit_all=False,       # after selecting the epoch, retrain on all 70 docs for the fold3 submission
)
METHOD_DEFAULTS = {
    "full":    dict(lr=5e-5),
    "partial": dict(lr=1e-4, train_top_layers=4),
    "lora":    dict(lr=1e-3, rank=16, alpha=32, targets=("qkv", "proj", "w12", "w3")),
    "lorma":   dict(lr=1e-3, rank=16, alpha=32, targets=("qkv", "proj", "w12", "w3")),
    "adapter": dict(lr=1e-3, adapter_dim=64),
    "student": dict(lr=1e-4, student_layers=None),   # None = every 2nd Phase-1 layer (6 of 12)
    "kd":      dict(lr=1e-4, student_layers=None, teacher="full", kd_alpha=0.5, kd_T=2.0),
}

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
TEACHER_DEVICE = torch.device("cuda:1") if torch.cuda.device_count() > 1 else DEVICE
os.makedirs(OUT_ROOT, exist_ok=True)
print("device:", DEVICE, "| teacher device:", TEACHER_DEVICE, "| GPUs:", torch.cuda.device_count())

device: cuda:0 | teacher device: cuda:1 | GPUs: 2


## 2. Locate inputs

In [3]:
def _walk(root="/kaggle/input"):
    if os.path.isdir(root):
        for dp, dn, fn in os.walk(root):
            yield dp, fn

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_PREFIX = "Phase_2_LNER"
HF_SECRET_NAME = "capstoned"                 # Kaggle secret holding your Hugging Face token

def download_from_hf():
    """Fallback when the Phase-2 files are not attached: download them from the course HF dataset."""
    from huggingface_hub import hf_hub_download, HfApi
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret(HF_SECRET_NAME)
        except Exception as e:
            raise RuntimeError(f"Phase-2 data not attached and no HF token found (Kaggle secret '{HF_SECRET_NAME}'): {e}")
    files = [f for f in HfApi().list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=token)
             if f.startswith(HF_PREFIX + "/")]
    if not files:
        raise FileNotFoundError(f"no files under {HF_PREFIX}/ in {HF_DATASET_REPO}")
    local = None
    for f in files:
        p = hf_hub_download(repo_id=HF_DATASET_REPO, filename=f, repo_type="dataset", token=token)
        print("  downloaded", f)
        if f.endswith("fold1-text.json"):
            local = os.path.dirname(os.path.dirname(p)) if os.path.basename(os.path.dirname(p)) == "Train" else os.path.dirname(p)
    return local

def find_data_dir():
    for dp, fn in _walk():
        if "fold1-text.json" in fn:
            return os.path.dirname(dp) if os.path.basename(dp) == "Train" else dp
    print("Phase-2 data not attached -> downloading from Hugging Face")
    return download_from_hf()

def find_file_named(name, root):
    for dp, fn in _walk(root):
        if name in fn:
            return os.path.join(dp, name)
    raise FileNotFoundError(f"{name} not found under {root}")

def find_legallm():
    hits = []
    for dp, fn in _walk():
        if {"config.json", "pytorch_model.bin"} <= set(fn):
            try:
                c = json.load(open(os.path.join(dp, "config.json")))
            except Exception:
                continue
            if {"n_layer", "d_model", "vocab_size", "max_seq_len"} <= set(c):
                hits.append(dp)
    if not hits:
        raise FileNotFoundError("Phase-1 LegalLM folder (config.json + pytorch_model.bin) not found")
    hits.sort(key=lambda p: ("legal" not in p.lower(), p))
    print("LegalLM folders found:", hits)
    return hits[0]

DATA_DIR = DATA_DIR or find_data_dir()
PHASE1_MODEL_DIR = PHASE1_MODEL_DIR or find_legallm()
if EVALUATOR is None:
    try:
        EVALUATOR = find_file_named("evaluate_phase2.py", "/kaggle/input")
    except FileNotFoundError:
        try:
            EVALUATOR = find_file_named("evaluate_phase2.py", DATA_DIR)
        except FileNotFoundError:
            EVALUATOR = ""
            print("evaluate_phase2.py not found -> using the built-in scorer (same strict rules)")

def data_file(name):
    for p in [os.path.join(DATA_DIR, "Train", name), os.path.join(DATA_DIR, "Test", name), os.path.join(DATA_DIR, name)]:
        if os.path.exists(p):
            return p
    return find_file_named(name, DATA_DIR)

TOKENIZER_PATH = os.path.join(PHASE1_MODEL_DIR, "tokenizer.json")
if not os.path.exists(TOKENIZER_PATH):
    TOKENIZER_PATH = find_file_named("tokenizer.json", "/kaggle/input")
LABELS_PATH = data_file("labels.txt")
print("data:", DATA_DIR, "\nmodel:", PHASE1_MODEL_DIR, "\ntokenizer:", TOKENIZER_PATH, "\nevaluator:", EVALUATOR)

Phase-2 data not attached -> downloading from Hugging Face


fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

  downloaded Phase_2_LNER/Test/fold3-text.json


fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

  downloaded Phase_2_LNER/Train/fold1-gold.json


fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

... (3 outputs left out)

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

  downloaded Phase_2_LNER/Train/fold2-text.json


labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

  downloaded Phase_2_LNER/Train/labels.txt
LegalLM folders found: ['/kaggle/input/datasets/adyansubeura/legal-lm-final']
data: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER 
model: /kaggle/input/datasets/adyansubeura/legal-lm-final 
tokenizer: /kaggle/input/datasets/adyansubeura/legal-lm-final/tokenizer.json 
evaluator: /kaggle/input/datasets/adyansubeura/evaluate-phase2/evaluate_phase2.py


## 3. Data, labels and the validation split

In [4]:
LABELS = [x.strip() for x in open(LABELS_PATH, encoding="utf-8") if x.strip()]
assert len(LABELS) == 12, LABELS
TAGS = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]
TAG2ID = {t: i for i, t in enumerate(TAGS)}
B_ID = {l: TAG2ID["B-" + l] for l in LABELS}
I_ID = {l: TAG2ID["I-" + l] for l in LABELS}
N_TAGS = len(TAGS)
IGNORE = -100

def load_json(p):
    with open(p, encoding="utf-8") as f:
        return json.load(f)

def load_fold(k, with_gold=True):
    sub = "Train" if with_gold else "Test"
    texts = load_json(data_file(f"fold{k}-text.json"))
    texts = {str(d): (v if isinstance(v, str) else v["text"]) for d, v in texts.items()}
    gold = None
    if with_gold:
        g = load_json(data_file(f"fold{k}-gold.json"))
        gold = {str(d): [tuple(e[:3]) for e in g.get(d, [])] for d in texts}
    if DEBUG_DOCS:
        keep = list(texts)[:DEBUG_DOCS]
        texts = {d: texts[d] for d in keep}
        gold = {d: gold[d] for d in keep} if gold else None
    return texts, gold

f1_text, f1_gold = load_fold(1)
f2_text, f2_gold = load_fold(2)
f3_text, _ = load_fold(3, with_gold=False)
ALL_TEXT = {**f1_text, **f2_text}
ALL_GOLD = {**f1_gold, **f2_gold}
for d, ents in ALL_GOLD.items():                       # offsets must slice the text exactly
    for s, e, l in ents:
        assert 0 <= s < e <= len(ALL_TEXT[d]) and l in LABELS, (d, s, e, l)

rng = random.Random(SEED)
VAL_IDS = []
for fold in (f1_text, f2_text):
    ids = sorted(fold)
    rng.shuffle(ids)
    VAL_IDS += ids[:min(VAL_DOCS_PER_FOLD, max(1, len(ids) // 3))]
TRAIN_IDS = [d for d in sorted(ALL_TEXT) if d not in set(VAL_IDS)]
TEST_IDS = sorted(f3_text)
print(f"train docs {len(TRAIN_IDS)} | val docs {len(VAL_IDS)} | test docs {len(TEST_IDS)}")
cnt = Counter(l for d in TRAIN_IDS for _, _, l in ALL_GOLD[d])
print("train entities per label:", dict(cnt.most_common()))
print("val entities per label:  ", dict(Counter(l for d in VAL_IDS for _, _, l in ALL_GOLD[d]).most_common()))

train docs 56 | val docs 14 | test docs 35
train entities per label: {'DATE': 1199, 'STAT': 1111, 'PREC': 1010, 'COURT': 842, 'AUTH': 668, 'CASENO': 636, 'RESP': 346, 'APP': 335, 'JUDGE': 202, 'WIT': 151, 'A.COUNSEL': 142, 'R.COUNSEL': 139}
val entities per label:   {'DATE': 376, 'AUTH': 335, 'PREC': 173, 'COURT': 171, 'STAT': 149, 'CASENO': 124, 'APP': 72, 'A.COUNSEL': 53, 'R.COUNSEL': 45, 'JUDGE': 43, 'WIT': 36, 'RESP': 35}


## 4. Tokenisation, character <-> token alignment, and the alignment ceiling

#### Each token's *core* span is its character span with surrounding whitespace removed (byte-level BPE glues the leading space to the token). Whitespace-only tokens are ignored for training and decoding, so an entity's predicted span always starts and ends on real characters. The **oracle** encodes the gold spans as tags and decodes them back: its score is the best any tagger can reach with this tokenizer.

In [5]:
from tokenizers import Tokenizer
tok = Tokenizer.from_file(TOKENIZER_PATH)
EOS_ID = tok.token_to_id("<|endoftext|>")
PAD_ID = EOS_ID if EOS_ID is not None else 0

class Doc:
    pass

def make_doc(did, text, gold):
    enc = tok.encode(text)
    d = Doc()
    d.id, d.text, d.ids = did, text, np.asarray(enc.ids, dtype=np.int64)
    core = []
    for (a, b) in enc.offsets:
        while a < b and text[a].isspace():
            a += 1
        while b > a and text[b - 1].isspace():
            b -= 1
        core.append((a, b))
    d.core = np.asarray(core, dtype=np.int64).reshape(-1, 2)
    d.real = np.nonzero(d.core[:, 1] > d.core[:, 0])[0]            # tokens with visible characters
    d.n = len(d.ids)
    d.gold = sorted(set(gold)) if gold is not None else None
    lab = np.full(d.n, IGNORE, dtype=np.int64)
    lab[d.real] = 0
    if gold:
        cs, ce = d.core[d.real, 0], d.core[d.real, 1]
        for s, e, l in sorted(gold):
            hit = d.real[(cs < e) & (ce > s)]
            if len(hit) == 0:
                continue
            lab[hit[0]] = B_ID[l]
            lab[hit[1:]] = I_ID[l]
    d.labels = lab
    return d

def windows(n, size, stride):
    out, s = [], 0
    while True:
        e = min(s + size, n)
        out.append((s, e))
        if e >= n:
            return out
        s += stride

# ---- constrained Viterbi over the real tokens of a document
_ALLOWED = np.ones((N_TAGS, N_TAGS), dtype=bool)
for l in LABELS:
    _ALLOWED[:, I_ID[l]] = False
    _ALLOWED[B_ID[l], I_ID[l]] = True
    _ALLOWED[I_ID[l], I_ID[l]] = True
_TRANS = np.where(_ALLOWED, 0.0, -1e9)
_START = np.array([-1e9 if t.startswith("I-") else 0.0 for t in TAGS])

def viterbi(logp):
    T = len(logp)
    if T == 0:
        return np.zeros(0, dtype=np.int64)
    back = np.zeros((T, N_TAGS), dtype=np.int64)
    dp = logp[0] + _START
    for t in range(1, T):
        sc = dp[:, None] + _TRANS
        back[t] = sc.argmax(0)
        dp = sc.max(0) + logp[t]
    path = np.zeros(T, dtype=np.int64)
    path[-1] = dp.argmax()
    for t in range(T - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path

def tags_to_spans(d, tag_seq):
    """tag_seq: one tag id per REAL token (d.real order) -> [[start, end, label], ...]"""
    spans, cur = [], None
    for k, t in enumerate(tag_seq):
        name = TAGS[t]
        if name == "O":
            if cur: spans.append(cur)
            cur = None
            continue
        p, l = name.split("-", 1)
        tokpos = d.real[k]
        if p == "B" or cur is None or cur[2] != l:
            if cur: spans.append(cur)
            cur = [int(d.core[tokpos, 0]), int(d.core[tokpos, 1]), l]
        else:
            cur[1] = int(d.core[tokpos, 1])
    if cur: spans.append(cur)
    return [list(x) for x in dict.fromkeys(tuple(x) for x in spans)]

# ---- strict scorer, identical rules to evaluate_phase2.py
def strict_scores(gold, pred):
    st = {l: [0, 0, 0] for l in LABELS}
    for did in gold:
        g = set(map(tuple, gold[did])); p = set(tuple(x[:3]) for x in pred.get(did, []))
        for l in LABELS:
            gs = {x for x in g if x[2] == l}; ps = {x for x in p if x[2] == l}
            st[l][0] += len(gs & ps); st[l][1] += len(ps - gs); st[l][2] += len(gs - ps)
    per, P, R, Fs = {}, [], [], []
    for l in LABELS:
        tp, fp, fn = st[l]
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        per[l] = {"p": pr, "r": rc, "f1": f, "tp": tp, "fp": fp, "fn": fn}
        P.append(pr); R.append(rc); Fs.append(f)
    return {"macro_f1": float(np.mean(Fs)), "macro_p": float(np.mean(P)), "macro_r": float(np.mean(R)), "per_label": per}

def official_eval(gold, pred, tag):
    if not EVALUATOR:                                                  # NEW
        return strict_scores(gold, pred)["macro_f1"], "(built-in scorer)"   # NEW
    tp_, pp_ = os.path.join(OUT_ROOT, f"_truth_{tag}.json"), os.path.join(OUT_ROOT, f"_pred_{tag}.json")
    json.dump({d: [list(x) for x in gold[d]] for d in gold}, open(tp_, "w"))
    json.dump({d: pred.get(d, []) for d in gold}, open(pp_, "w"))
    r = subprocess.run([sys.executable, EVALUATOR, "--truth", tp_, "--prediction", pp_, "--labels", LABELS_PATH],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("official evaluator failed: " + r.stderr + r.stdout)
    return float(re.search(r"Strict Macro-F1:\s+([\d.]+)", r.stdout).group(1)), r.stdout

t0 = time.time()
DOCS = {d: make_doc(d, ALL_TEXT[d], ALL_GOLD[d]) for d in ALL_TEXT}
TEST_DOCS = {d: make_doc(d, f3_text[d], None) for d in TEST_IDS}
print(f"tokenised {len(DOCS) + len(TEST_DOCS)} docs in {time.time() - t0:.1f}s | tokens/doc median "
      f"{np.median([x.n for x in DOCS.values()]):.0f}, max {max(x.n for x in DOCS.values())}")

oracle = {d: tags_to_spans(x, x.labels[x.real]) for d, x in DOCS.items()}
gold_all = {d: DOCS[d].gold for d in DOCS}
o = strict_scores(gold_all, oracle)
o_off, _ = official_eval(gold_all, oracle, "oracle")
print(f"ORACLE strict macro-F1 (gold -> tags -> spans): {o['macro_f1']:.4f} (official evaluator: {o_off:.4f})")
print("   per label:", {l: round(v["f1"], 3) for l, v in o["per_label"].items()})
lost = [(d, s, e, l, ALL_TEXT[d][s:e]) for d in DOCS for (s, e, l) in DOCS[d].gold if (s, e, l) not in set(map(tuple, oracle[d]))]
print(f"   {len(lost)} gold spans cannot be produced by any token tagger, e.g.:", [x[3:] for x in lost[:6]])

tokenised 105 docs in 2.5s | tokens/doc median 5554, max 23999
ORACLE strict macro-F1 (gold -> tags -> spans): 0.9865 (official evaluator: 0.9865)
   per label: {'APP': 0.983, 'RESP': 0.982, 'A.COUNSEL': 0.995, 'R.COUNSEL': 1.0, 'JUDGE': 1.0, 'WIT': 1.0, 'AUTH': 0.986, 'COURT': 0.982, 'STAT': 0.98, 'PREC': 0.932, 'DATE': 1.0, 'CASENO': 0.999}
   154 gold spans cannot be produced by any token tagger, e.g.: [('PREC', 'State of M.P. vs. Kashiram & Ors.'), ('PREC', 'Gambangi Appalaswamy Naldu and Others v. Behara Venkataramanayya Patro [AIR 1984- SC 1728]'), ('AUTH', 'Railway Board'), ('AUTH', 'Railway Board'), ('PREC', 'Shankarsan Dash Vs. Union of India [1991 (3) SCC 47]'), ('PREC', 'Jatendra Kumar vs. State of Punjab [(1985) 1 SCC 122]')]


## 5. Model: Phase-1 LegalLM as an encoder, plus LoRA / LoRMA / adapters

#### Same weights and module names as the Phase-1 `model.py`, so `pytorch_model.bin` loads directly. `bidirectional=True` drops the causal mask; padding is masked either way.

In [6]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight).to(x.dtype)

def rope_tables(cfg):
    hd = cfg["d_model"] // cfg["n_head"]
    inv = 1.0 / (cfg.get("rope_theta", 10000.0) ** (torch.arange(0, hd, 2).float() / hd))
    f = torch.outer(torch.arange(cfg["max_seq_len"]).float(), inv)
    return f.cos(), f.sin()

def apply_rope(x, cos, sin):
    h = x.shape[-1] // 2
    x1, x2 = x[..., :h], x[..., h:]
    cos, sin = cos.to(x.dtype), sin.to(x.dtype)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)

class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h, self.hd = cfg["n_head"], cfg["d_model"] // cfg["n_head"]
        self.qkv = nn.Linear(cfg["d_model"], 3 * cfg["d_model"], bias=False)
        self.proj = nn.Linear(cfg["d_model"], cfg["d_model"], bias=False)
    def forward(self, x, cos, sin, mask, causal):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.h, self.hd).transpose(1, 2) for t in (q, k, v))
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if causal:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)   # right padding never reaches real tokens
        else:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)  # mask: (B,1,1,T) True = attend
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w12 = nn.Linear(cfg["d_model"], 2 * cfg["d_ff"], bias=False)
        self.w3 = nn.Linear(cfg["d_ff"], cfg["d_model"], bias=False)
    def forward(self, x):
        a, b = self.w12(x).chunk(2, dim=-1)
        return self.w3(F.silu(a) * b)

class Adapter(nn.Module):
    """Pfeiffer-style bottleneck adapter after the FFN; zero-initialised up-projection = identity at start."""
    def __init__(self, d, r):
        super().__init__()
        self.down, self.up = nn.Linear(d, r), nn.Linear(r, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg["d_model"], cfg.get("norm_eps", 1e-5))
        self.attn = Attention(cfg)
        self.norm2 = RMSNorm(cfg["d_model"], cfg.get("norm_eps", 1e-5))
        self.mlp = SwiGLU(cfg)
        self.adapter = None
    def forward(self, x, cos, sin, mask, causal):
        x = x + self.attn(self.norm1(x), cos, sin, mask, causal)
        h = self.mlp(self.norm2(x))
        if self.adapter is not None:
            h = self.adapter(h)
        return x + h

class LegalEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg["n_layer"])])
        self.norm = RMSNorm(cfg["d_model"], cfg.get("norm_eps", 1e-5))
        cos, sin = rope_tables(cfg)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
    def forward(self, ids, key_mask, causal):
        T = ids.shape[1]
        cos, sin = self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]
        mask = None if causal else key_mask[:, None, None, :]
        x = self.tok(ids)
        for b in self.blocks:
            x = b(x, cos, sin, mask, causal)
        return self.norm(x)

class NERModel(nn.Module):
    def __init__(self, cfg, dropout, bidirectional):
        super().__init__()
        self.encoder = LegalEncoder(cfg)
        self.causal = not bidirectional
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(cfg["d_model"], N_TAGS)
        nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)
    def forward(self, ids, key_mask):
        return self.head(self.drop(self.encoder(ids, key_mask, self.causal)))

class LoRALinear(nn.Module):
    """W0 x + (alpha/r) B A x   (Hu et al.)  - additive low-rank update."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        return self.base(x) + self.s * F.linear(F.linear(x, self.A), self.B)

class LoRMALinear(nn.Module):
    """LoRMA+ (Bihany et al., 2025): (I + (alpha/r) B A) W0 x  - multiplicative low-rank update of the
    pretrained weight. Computed as y0 + s*B(A y0) with y0 = W0 x, so BA (out x out) is never formed.
    B = 0 at init -> starts exactly at the pretrained model."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        y0 = self.base(x)
        return y0 + self.s * F.linear(F.linear(y0, self.A), self.B)

def wrap_linears(model, cls, targets, r, alpha):
    n = 0
    for blk in model.encoder.blocks:
        for parent, name in [(blk.attn, "qkv"), (blk.attn, "proj"), (blk.mlp, "w12"), (blk.mlp, "w3")]:
            if name in targets:
                setattr(parent, name, cls(getattr(parent, name), r, alpha)); n += 1
    return n

def load_phase1_into(model, model_dir, layer_map=None):
    sd = torch.load(os.path.join(model_dir, "pytorch_model.bin"), map_location="cpu")
    new = {}
    for k, v in sd.items():
        if k.startswith("head."):
            continue                                       # LM head (tied to tok) is not used for NER
        if k.startswith("blocks.") and layer_map is not None:
            i = int(k.split(".")[1])
            if i not in layer_map:
                continue
            k = "blocks." + str(layer_map[i]) + "." + k.split(".", 2)[2]
        new["encoder." + k] = v
    missing, unexpected = model.load_state_dict(new, strict=False)
    missing = [m for m in missing if not m.startswith("head.")]
    assert not missing and not unexpected, (missing[:5], unexpected[:5])

def build_model(method, hp):
    cfg = dict(json.load(open(os.path.join(PHASE1_MODEL_DIR, "config.json"))))
    layer_map = None
    if method in ("student", "kd"):
        layers = list(hp["student_layers"]) if hp.get("student_layers") else list(range(0, cfg["n_layer"], 2))
        assert all(0 <= i < cfg["n_layer"] for i in layers), f"student_layers must be within 0..{cfg['n_layer'] - 1}"
        hp["student_layers"] = layers
        layer_map = {src: dst for dst, src in enumerate(layers)}
        cfg["n_layer"] = len(layers)
    model = NERModel(cfg, hp["dropout"], hp["bidirectional"])
    load_phase1_into(model, PHASE1_MODEL_DIR, layer_map)
    enc = model.encoder
    if method in ("lora", "lorma", "adapter", "partial"):
        for p in enc.parameters():
            p.requires_grad = False
    if method == "lora":
        wrap_linears(model, LoRALinear, hp["targets"], hp["rank"], hp["alpha"])
    elif method == "lorma":
        wrap_linears(model, LoRMALinear, hp["targets"], hp["rank"], hp["alpha"])
    elif method == "adapter":
        for blk in enc.blocks:
            blk.adapter = Adapter(cfg["d_model"], hp["adapter_dim"])
    elif method == "partial":
        k = hp["train_top_layers"]
        for blk in enc.blocks[-k:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in enc.norm.parameters():
            p.requires_grad = True
    return model, cfg

def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr

## 6. Training, window merging and prediction

In [7]:
def make_windows(docs, size, stride):
    out = []
    for d in docs:
        for (s, e) in windows(d.n, size, stride):
            out.append((d, s, e))
    return out

def collate(items, with_labels=True):
    L = max(e - s for _, s, e in items)
    ids = torch.full((len(items), L), PAD_ID, dtype=torch.long)
    km = torch.zeros((len(items), L), dtype=torch.bool)
    lab = torch.full((len(items), L), IGNORE, dtype=torch.long)
    for i, (d, s, e) in enumerate(items):
        ids[i, :e - s] = torch.from_numpy(d.ids[s:e]); km[i, :e - s] = True
        if with_labels and d.labels is not None:
            lab[i, :e - s] = torch.from_numpy(d.labels[s:e])
    return ids, km, lab

def class_weights(train_docs, hp):
    w = np.ones(N_TAGS)
    if hp["loss"] == "weighted":
        c = np.bincount(np.concatenate([d.labels[d.labels >= 0] for d in train_docs]), minlength=N_TAGS).astype(float)
        for l in LABELS:                                    # same weight for B-X and I-X: sqrt inverse frequency
            n = c[B_ID[l]] + c[I_ID[l]]
            w[B_ID[l]] = w[I_ID[l]] = math.sqrt(c.sum() / max(n, 1.0))
        w[0] = math.sqrt(c.sum() / max(c[0], 1.0))
        w = w / w[0]
    w[0] *= hp["o_weight"]
    return torch.tensor(w, dtype=torch.float32)

@torch.no_grad()
def predict_docs(model, docs, hp, device=DEVICE, return_probs=False):
    """Sliding windows -> centre-weighted average of probabilities -> constrained Viterbi -> spans."""
    model.eval()
    amp = torch.autocast("cuda", dtype=torch.float16) if (hp["fp16"] and device.type == "cuda") else contextlib.nullcontext()
    items = make_windows(docs, hp["window"], hp["stride"])
    acc = {d.id: np.zeros((d.n, N_TAGS), dtype=np.float32) for d in docs}
    wsum = {d.id: np.zeros(d.n, dtype=np.float32) for d in docs}
    for k in range(0, len(items), hp["eval_batch_size"]):
        batch = items[k:k + hp["eval_batch_size"]]
        ids, km, _ = collate(batch, with_labels=False)
        with amp:
            logits = model(ids.to(device), km.to(device))
        probs = torch.softmax(logits.float(), -1).cpu().numpy()
        for i, (d, s, e) in enumerate(batch):
            pos = np.arange(s, e)
            if hp["bidirectional"]:
                left = pos - s if s > 0 else np.full(e - s, 10 ** 6)
                right = (e - 1) - pos if e < d.n else np.full(e - s, 10 ** 6)
                w = np.minimum(np.minimum(left, right), hp["window"]) + 1.0
            else:
                w = (pos - s) + 1.0 if s > 0 else np.full(e - s, float(hp["window"]))   # more left context = better
            acc[d.id][s:e] += probs[i, :e - s] * w[:, None]
            wsum[d.id][s:e] += w
    out, probs_out = {}, {}
    for d in docs:
        p = acc[d.id][d.real] / np.maximum(wsum[d.id][d.real], 1e-9)[:, None]
        out[d.id] = tags_to_spans(d, viterbi(np.log(p + 1e-9)))
        if return_probs:
            probs_out[d.id] = p
    return (out, probs_out) if return_probs else out

def load_teacher(name):
    tdir = os.path.join(OUT_ROOT, name)
    meta = json.load(open(os.path.join(tdir, "run_config.json")))
    thp = meta["hp"]
    teacher, _ = build_model(meta["method"], thp)
    teacher.load_state_dict(torch.load(os.path.join(tdir, "best_model.pt"), map_location="cpu"))
    teacher = teacher.to(TEACHER_DEVICE).eval()
    for p in teacher.parameters():
        p.requires_grad = False
    print(f"teacher '{name}' ({meta['method']}, {count_params(teacher)[0] / 1e6:.1f}M params) on {TEACHER_DEVICE}")
    return teacher, thp

def train_model(model, hp, method, train_docs, val_docs, epochs, out_dir, select=True, teacher=None):
    model = model.to(DEVICE)
    head_params = list(model.head.parameters())
    head_ids = {id(p) for p in head_params}
    body = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    decay = [p for p in body if p.ndim >= 2]
    no_decay = [p for p in body if p.ndim < 2]
    opt = torch.optim.AdamW([{"params": decay, "lr": hp["lr"], "weight_decay": hp["weight_decay"]},
                             {"params": no_decay, "lr": hp["lr"], "weight_decay": 0.0},
                             {"params": head_params, "lr": hp["head_lr"], "weight_decay": 0.0}])
    items = make_windows(train_docs, hp["window"], hp["stride"])
    steps_per_epoch = math.ceil(len(items) / hp["batch_size"])
    total = steps_per_epoch * epochs
    warm = max(1, int(hp["warmup"] * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * max(0.0, (total - s) / max(1, total - warm)))
    use_amp = hp["fp16"] and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    cw = class_weights(train_docs, hp).to(DEVICE)
    rng = random.Random(SEED)
    history, best, bad = [], None, 0
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)
    t_train = 0.0
    for ep in range(1, epochs + 1):
        model.train()
        rng.shuffle(items)
        t0 = time.time()
        tot_loss, nb = 0.0, 0
        for k in range(0, len(items), hp["batch_size"]):
            ids, km, lab = collate(items[k:k + hp["batch_size"]])
            ids, km, lab = ids.to(DEVICE), km.to(DEVICE), lab.to(DEVICE)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(ids, km)
            logits = logits.float()
            valid = lab != IGNORE
            loss = F.cross_entropy(logits[valid], lab[valid], weight=cw)
            if teacher is not None:
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=TEACHER_DEVICE.type == "cuda"):
                    t_logits = teacher(ids.to(TEACHER_DEVICE), km.to(TEACHER_DEVICE))
                t_logits = t_logits.float().to(DEVICE)[valid]
                T = hp["kd_T"]
                kd = F.kl_div(F.log_softmax(logits[valid] / T, -1), F.softmax(t_logits / T, -1), reduction="batchmean") * T * T
                loss = (1 - hp["kd_alpha"]) * loss + hp["kd_alpha"] * kd
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], hp["grad_clip"])
            scaler.step(opt); scaler.update(); sched.step()
            tot_loss += loss.item(); nb += 1
        t_train += time.time() - t0
        rec = {"epoch": ep, "train_loss": tot_loss / max(nb, 1), "epoch_time_s": time.time() - t0}
        if select:
            t1 = time.time()
            preds = predict_docs(model, val_docs, hp)
            sc = strict_scores({d.id: d.gold for d in val_docs}, preds)
            rec.update({"val_macro_f1": sc["macro_f1"], "val_macro_p": sc["macro_p"], "val_macro_r": sc["macro_r"],
                        "val_time_s": time.time() - t1})
            improved = best is None or sc["macro_f1"] > best["val_macro_f1"]
            if improved:
                best = {**rec, "per_label": sc["per_label"], "preds": preds}
                torch.save(model.state_dict(), os.path.join(out_dir, "best_model.pt"))
                bad = 0
            else:
                bad += 1
            print(f"  ep {ep:>2}/{epochs}  loss {rec['train_loss']:.4f}  val macro-F1 {sc['macro_f1']:.4f} "
                  f"(P {sc['macro_p']:.3f} R {sc['macro_r']:.3f}){'  *' if improved else ''}  [{rec['epoch_time_s']:.0f}s]", flush=True)
            history.append(rec)
            if bad >= hp["patience"]:
                print(f"  early stop: no improvement for {hp['patience']} epochs")
                break
        else:
            print(f"  ep {ep:>2}/{epochs}  loss {rec['train_loss']:.4f}  [{rec['epoch_time_s']:.0f}s]", flush=True)
            history.append(rec)
    if not select:
        torch.save(model.state_dict(), os.path.join(out_dir, "best_model.pt"))
    peak = torch.cuda.max_memory_allocated(DEVICE) / 2 ** 30 if DEVICE.type == "cuda" else None
    return history, best, t_train, peak

## 7. Run one experiment (train -> select epoch on validation -> fold3 submission)

In [8]:
def write_submission(preds, path_zip):
    pj = os.path.join(os.path.dirname(path_zip), "prediction.json")
    clean = {d: [[int(s), int(e), l] for s, e, l in preds.get(d, [])] for d in TEST_IDS}
    json.dump(clean, open(pj, "w"), ensure_ascii=False)
    with zipfile.ZipFile(path_zip, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    return clean

def run_experiment(name, method, **overrides):
    assert method in METHOD_DEFAULTS, method
    hp = {**DEFAULTS, **METHOD_DEFAULTS[method], **overrides}
    out_dir = os.path.join(OUT_ROOT, name)
    os.makedirs(out_dir, exist_ok=True)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    print("=" * 100 + f"\n{name}  |  method={method}\n  hp={json.dumps({k: (list(v) if isinstance(v, tuple) else v) for k, v in hp.items()})}")
    model, cfg = build_model(method, hp)
    total, trainable = count_params(model)
    print(f"  params total {total / 1e6:.2f}M | trainable {trainable / 1e6:.3f}M ({100 * trainable / total:.2f}%) | "
          f"layers {cfg['n_layer']} | {'bidirectional' if hp['bidirectional'] else 'causal'}")
    json.dump({"method": method, "hp": hp, "model_cfg": cfg}, open(os.path.join(out_dir, "run_config.json"), "w"), indent=1)
    teacher = None
    if method == "kd":
        teacher, thp = load_teacher(hp["teacher"])
        assert thp["window"] == hp["window"] and thp["bidirectional"] == hp["bidirectional"], "teacher/student window setup must match"

    train_docs = [DOCS[d] for d in TRAIN_IDS]
    val_docs = [DOCS[d] for d in VAL_IDS]
    history, best, t_train, peak = train_model(model, hp, method, train_docs, val_docs, hp["epochs"], out_dir, teacher=teacher)
    off_f1, off_txt = official_eval({d.id: d.gold for d in val_docs}, best["preds"], name)
    print(f"  BEST epoch {best['epoch']}: val strict macro-F1 {best['val_macro_f1']:.4f} (official evaluator {off_f1:.4f})")

    # fold3 predictions with the selected checkpoint
    model.load_state_dict(torch.load(os.path.join(out_dir, "best_model.pt"), map_location=DEVICE))
    t0 = time.time()
    test_preds = predict_docs(model, [TEST_DOCS[d] for d in TEST_IDS], hp)
    infer_s = time.time() - t0
    write_submission(test_preds, os.path.join(out_dir, f"submission_{name}.zip"))

    refit = None
    if hp["refit_all"]:
        print(f"  refit on all {len(TRAIN_IDS) + len(VAL_IDS)} docs for {best['epoch']} epochs")
        random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
        model2, _ = build_model(method, hp)
        rdir = os.path.join(out_dir, "refit"); os.makedirs(rdir, exist_ok=True)
        train_model(model2, hp, method, train_docs + val_docs, [], best["epoch"], rdir, select=False, teacher=teacher)
        write_submission(predict_docs(model2, [TEST_DOCS[d] for d in TEST_IDS], hp), os.path.join(out_dir, f"submission_{name}_refit.zip"))
        refit = True
        del model2

    metrics = {"name": name, "method": method, "params_total": total, "params_trainable": trainable,
               "layers": cfg["n_layer"], "bidirectional": hp["bidirectional"],
               "best_epoch": best["epoch"], "val_macro_f1": best["val_macro_f1"], "val_macro_f1_official": off_f1,
               "val_macro_p": best["val_macro_p"], "val_macro_r": best["val_macro_r"],
               "per_label_f1": {l: v["f1"] for l, v in best["per_label"].items()},
               "per_label": best["per_label"], "train_time_s": t_train, "peak_gpu_gb": peak,
               "test_infer_s": infer_s, "test_infer_ms_per_doc": 1000 * infer_s / max(1, len(TEST_IDS)),
               "checkpoint_mb": os.path.getsize(os.path.join(out_dir, "best_model.pt")) / 2 ** 20,
               "history": history, "hp": hp, "refit": refit}
    json.dump(metrics, open(os.path.join(out_dir, "metrics.json"), "w"), indent=1)
    json.dump(best["preds"], open(os.path.join(out_dir, "val_predictions.json"), "w"))
    print(f"  wrote {out_dir}/submission_{name}.zip and metrics.json")
    del model, teacher
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return metrics

## 8. Comparison table (all finished experiments)

In [9]:
def summary():
    rows = []
    for p in sorted(glob.glob(os.path.join(OUT_ROOT, "*", "metrics.json"))):
        m = json.load(open(p))
        rows.append(m)
    if not rows:
        print("no finished experiments yet"); return
    hdr = f"{'run':<16}{'method':<9}{'val F1':>8}{'P':>7}{'R':>7}{'params':>9}{'trainable':>11}{'train min':>10}{'GPU GB':>8}{'ms/doc':>8}"
    print(hdr); print("-" * len(hdr))
    for m in rows:
        print(f"{m['name']:<16}{m['method']:<9}{m['val_macro_f1']:>8.4f}{m['val_macro_p']:>7.3f}{m['val_macro_r']:>7.3f}"
              f"{m['params_total'] / 1e6:>8.1f}M{m['params_trainable'] / 1e6:>10.3f}M{m['train_time_s'] / 60:>10.1f}"
              f"{(m['peak_gpu_gb'] or 0):>8.2f}{m['test_infer_ms_per_doc']:>8.0f}")
    print("\nper-label F1:")
    print(f"{'label':<11}" + "".join(f"{m['name'][:12]:>13}" for m in rows))
    for l in LABELS:
        print(f"{l:<11}" + "".join(f"{m['per_label_f1'][l]:>13.3f}" for m in rows))

## 9. Run the experiments

#### Order matters only for `kd`, which needs the finished `full` run as its teacher. Each run prints validation macro-F1 per epoch and writes `submission_<name>.zip`. To tune, call `run_experiment` again with a new name and overrides, e.g. `run_experiment("lora_r32", "lora", rank=32, alpha=64)`.

In [12]:
results = {}
results["full"] = run_experiment("full", "full")

full  |  method=full
  hp={"bidirectional": true, "window": 512, "stride": 384, "epochs": 20, "patience": 6, "batch_size": 8, "eval_batch_size": 32, "lr": 5e-05, "head_lr": 0.001, "weight_decay": 0.01, "warmup": 0.1, "grad_clip": 1.0, "dropout": 0.1, "loss": "ce", "o_weight": 1.0, "fp16": true, "refit_all": false}
  params total 109.55M | trainable 109.549M (100.00%) | layers 12 | bidirectional


/tmp/ipykernel_49/2279519611.py:122: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


  ep  1/20  loss 0.9701  val macro-F1 0.2364 (P 0.226 R 0.253)  *  [32s]
  ep  2/20  loss 0.1030  val macro-F1 0.3911 (P 0.383 R 0.417)  *  [31s]
  ep  3/20  loss 0.0501  val macro-F1 0.3766 (P 0.477 R 0.375)  [32s]
  ep  4/20  loss 0.0229  val macro-F1 0.4260 (P 0.413 R 0.456)  *  [33s]
  ep  5/20  loss 0.0135  val macro-F1 0.4346 (P 0.425 R 0.470)  *  [32s]
  ep  6/20  loss 0.0065  val macro-F1 0.4505 (P 0.447 R 0.475)  *  [32s]
  ep  7/20  loss 0.0023  val macro-F1 0.4302 (P 0.417 R 0.472)  [32s]
  ep  8/20  loss 0.0012  val macro-F1 0.4327 (P 0.452 R 0.449)  [32s]
  ep  9/20  loss 0.0010  val macro-F1 0.4620 (P 0.450 R 0.496)  *  [32s]
  ep 10/20  loss 0.0004  val macro-F1 0.4365 (P 0.465 R 0.449)  [32s]
  ep 11/20  loss 0.0001  val macro-F1 0.4432 (P 0.453 R 0.464)  [32s]
  ep 12/20  loss 0.0001  val macro-F1 0.4422 (P 0.451 R 0.463)  [32s]
  ep 13/20  loss 0.0001  val macro-F1 0.4417 (P 0.448 R 0.463)  [32s]
  ep 14/20  loss 0.0000  val macro-F1 0.4414 (P 0.448 R 0.462)  [32s]
  

In [13]:
results["lora"] = run_experiment("lora", "lora")

lora  |  method=lora
  hp={"bidirectional": true, "window": 512, "stride": 384, "epochs": 20, "patience": 6, "batch_size": 8, "eval_batch_size": 32, "lr": 0.001, "head_lr": 0.001, "weight_decay": 0.01, "warmup": 0.1, "grad_clip": 1.0, "dropout": 0.1, "loss": "ce", "o_weight": 1.0, "fp16": true, "refit_all": false, "rank": 16, "alpha": 32, "targets": ["qkv", "proj", "w12", "w3"]}
  params total 111.91M | trainable 2.379M (2.13%) | layers 12 | bidirectional


/tmp/ipykernel_49/2279519611.py:122: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


  ep  1/20  loss 0.9278  val macro-F1 0.2605 (P 0.271 R 0.275)  *  [29s]
  ep  2/20  loss 0.1049  val macro-F1 0.4160 (P 0.401 R 0.459)  *  [29s]
  ep  3/20  loss 0.0693  val macro-F1 0.4209 (P 0.535 R 0.392)  *  [29s]
  ep  4/20  loss 0.0409  val macro-F1 0.4798 (P 0.500 R 0.489)  *  [29s]
  ep  5/20  loss 0.0241  val macro-F1 0.4854 (P 0.538 R 0.490)  *  [29s]
  ep  6/20  loss 0.0167  val macro-F1 0.5135 (P 0.528 R 0.539)  *  [29s]
  ep  7/20  loss 0.0090  val macro-F1 0.5135 (P 0.526 R 0.520)  *  [29s]
  ep  8/20  loss 0.0091  val macro-F1 0.4820 (P 0.536 R 0.474)  [29s]
  ep  9/20  loss 0.0043  val macro-F1 0.5229 (P 0.531 R 0.535)  *  [29s]
  ep 10/20  loss 0.0030  val macro-F1 0.5142 (P 0.540 R 0.523)  [29s]
  ep 11/20  loss 0.0018  val macro-F1 0.5141 (P 0.527 R 0.525)  [29s]
  ep 12/20  loss 0.0008  val macro-F1 0.5245 (P 0.544 R 0.527)  *  [29s]
  ep 13/20  loss 0.0002  val macro-F1 0.5227 (P 0.532 R 0.536)  [29s]
  ep 14/20  loss 0.0001  val macro-F1 0.5182 (P 0.527 R 0.530) 

In [14]:
results["lorma"] = run_experiment("lorma", "lorma")

lorma  |  method=lorma
  hp={"bidirectional": true, "window": 512, "stride": 384, "epochs": 20, "patience": 6, "batch_size": 8, "eval_batch_size": 32, "lr": 0.001, "head_lr": 0.001, "weight_decay": 0.01, "warmup": 0.1, "grad_clip": 1.0, "dropout": 0.1, "loss": "ce", "o_weight": 1.0, "fp16": true, "refit_all": false, "rank": 16, "alpha": 32, "targets": ["qkv", "proj", "w12", "w3"]}
  params total 112.60M | trainable 3.067M (2.72%) | layers 12 | bidirectional


/tmp/ipykernel_49/2279519611.py:122: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


  ep  1/20  loss 0.7234  val macro-F1 0.3940 (P 0.403 R 0.405)  *  [30s]
  ep  2/20  loss 0.1066  val macro-F1 0.3097 (P 0.346 R 0.306)  [30s]
  ep  3/20  loss 0.0956  val macro-F1 0.4262 (P 0.440 R 0.434)  *  [29s]
  ep  4/20  loss 0.0770  val macro-F1 0.4566 (P 0.452 R 0.497)  *  [30s]
  ep  5/20  loss 0.0603  val macro-F1 0.4017 (P 0.443 R 0.397)  [30s]
  ep  6/20  loss 0.0492  val macro-F1 0.4688 (P 0.480 R 0.525)  *  [30s]
  ep  7/20  loss 0.0390  val macro-F1 0.4096 (P 0.424 R 0.427)  [30s]
  ep  8/20  loss 0.0285  val macro-F1 0.4257 (P 0.465 R 0.430)  [29s]
  ep  9/20  loss 0.0271  val macro-F1 0.4893 (P 0.505 R 0.488)  *  [30s]
  ep 10/20  loss 0.0202  val macro-F1 0.4348 (P 0.474 R 0.425)  [29s]
  ep 11/20  loss 0.0111  val macro-F1 0.5414 (P 0.581 R 0.545)  *  [29s]
  ep 12/20  loss 0.0069  val macro-F1 0.5005 (P 0.525 R 0.504)  [29s]
  ep 13/20  loss 0.0059  val macro-F1 0.5380 (P 0.538 R 0.554)  [29s]
  ep 14/20  loss 0.0037  val macro-F1 0.5125 (P 0.526 R 0.519)  [29s]
  

In [15]:
results["student"] = run_experiment("student", "student")

student  |  method=student
  hp={"bidirectional": true, "window": 512, "stride": 384, "epochs": 20, "patience": 6, "batch_size": 8, "eval_batch_size": 32, "lr": 0.0001, "head_lr": 0.001, "weight_decay": 0.01, "warmup": 0.1, "grad_clip": 1.0, "dropout": 0.1, "loss": "ce", "o_weight": 1.0, "fp16": true, "refit_all": false, "student_layers": null}
  params total 67.07M | trainable 67.073M (100.00%) | layers 6 | bidirectional


/tmp/ipykernel_49/2279519611.py:122: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


  ep  1/20  loss 0.7449  val macro-F1 0.2602 (P 0.277 R 0.271)  *  [17s]
  ep  2/20  loss 0.1011  val macro-F1 0.3891 (P 0.375 R 0.442)  *  [17s]
  ep  3/20  loss 0.0514  val macro-F1 0.3277 (P 0.376 R 0.315)  [17s]
  ep  4/20  loss 0.0235  val macro-F1 0.4221 (P 0.424 R 0.447)  *  [17s]
  ep  5/20  loss 0.0112  val macro-F1 0.4233 (P 0.425 R 0.442)  *  [17s]
  ep  6/20  loss 0.0076  val macro-F1 0.4516 (P 0.463 R 0.459)  *  [17s]
  ep  7/20  loss 0.0033  val macro-F1 0.4552 (P 0.450 R 0.477)  *  [17s]
  ep  8/20  loss 0.0015  val macro-F1 0.4405 (P 0.447 R 0.449)  [17s]
  ep  9/20  loss 0.0006  val macro-F1 0.4473 (P 0.460 R 0.451)  [17s]
  ep 10/20  loss 0.0002  val macro-F1 0.4504 (P 0.458 R 0.458)  [17s]
  ep 11/20  loss 0.0001  val macro-F1 0.4494 (P 0.457 R 0.457)  [17s]
  ep 12/20  loss 0.0001  val macro-F1 0.4532 (P 0.461 R 0.460)  [17s]
  ep 13/20  loss 0.0001  val macro-F1 0.4527 (P 0.460 R 0.460)  [17s]
  early stop: no improvement for 6 epochs
  BEST epoch 7: val strict mac

In [16]:
results["kd"] = run_experiment("kd", "kd", teacher="full")

kd  |  method=kd
  hp={"bidirectional": true, "window": 512, "stride": 384, "epochs": 20, "patience": 6, "batch_size": 8, "eval_batch_size": 32, "lr": 0.0001, "head_lr": 0.001, "weight_decay": 0.01, "warmup": 0.1, "grad_clip": 1.0, "dropout": 0.1, "loss": "ce", "o_weight": 1.0, "fp16": true, "refit_all": false, "student_layers": null, "teacher": "full", "kd_alpha": 0.5, "kd_T": 2.0}
  params total 67.07M | trainable 67.073M (100.00%) | layers 6 | bidirectional
teacher 'full' (full, 109.5M params) on cuda:1


/tmp/ipykernel_49/2279519611.py:122: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


  ep  1/20  loss 1.9853  val macro-F1 0.2355 (P 0.283 R 0.245)  *  [26s]
  ep  2/20  loss 0.2439  val macro-F1 0.4313 (P 0.417 R 0.461)  *  [26s]
  ep  3/20  loss 0.1231  val macro-F1 0.3368 (P 0.369 R 0.335)  [26s]
  ep  4/20  loss 0.0612  val macro-F1 0.4290 (P 0.443 R 0.444)  [26s]
  ep  5/20  loss 0.0346  val macro-F1 0.4392 (P 0.449 R 0.449)  *  [26s]
  ep  6/20  loss 0.0204  val macro-F1 0.4764 (P 0.484 R 0.492)  *  [26s]
  ep  7/20  loss 0.0129  val macro-F1 0.4889 (P 0.486 R 0.507)  *  [26s]
  ep  8/20  loss 0.0093  val macro-F1 0.4954 (P 0.498 R 0.506)  *  [26s]
  ep  9/20  loss 0.0075  val macro-F1 0.4871 (P 0.492 R 0.497)  [26s]
  ep 10/20  loss 0.0063  val macro-F1 0.4924 (P 0.502 R 0.501)  [26s]
  ep 11/20  loss 0.0058  val macro-F1 0.4900 (P 0.495 R 0.500)  [26s]
  ep 12/20  loss 0.0054  val macro-F1 0.4975 (P 0.504 R 0.507)  *  [26s]
  ep 13/20  loss 0.0051  val macro-F1 0.5001 (P 0.509 R 0.509)  *  [26s]
  ep 14/20  loss 0.0049  val macro-F1 0.5019 (P 0.505 R 0.513)  * 

In [17]:
summary()

run             method     val F1      P      R   params  trainable train min  GPU GB  ms/doc
---------------------------------------------------------------------------------------------
full            full       0.4620  0.450  0.496   109.5M   109.549M       8.0    3.78     236
kd              kd         0.5019  0.505  0.513    67.1M    67.073M       8.7    2.11     159
lora            lora       0.5245  0.544  0.527   111.9M     2.379M       8.7    2.28     289
lorma           lorma      0.5414  0.581  0.545   112.6M     3.067M       8.3    2.69     287
student         student    0.4552  0.450  0.477    67.1M    67.073M       3.7    2.11     161

per-label F1:
label               full           kd         lora        lorma      student
APP                0.182        0.268        0.234        0.286        0.191
RESP               0.354        0.272        0.380        0.429        0.247
A.COUNSEL          0.250        0.455        0.440        0.738        0.384
R.COUNSEL          

In [10]:
results["lorma_causal"] = run_experiment("lorma_causal", "lorma", bidirectional=False)
results["lorma_long"]   = run_experiment("lorma_long", "lorma", epochs=40, patience=10)
results["lorma_wce"]    = run_experiment("lorma_wce", "lorma", epochs=40, patience=10, loss="weighted")
summary()

lorma_causal  |  method=lorma
  hp={"bidirectional": false, "window": 512, "stride": 384, "epochs": 20, "patience": 6, "batch_size": 8, "eval_batch_size": 32, "lr": 0.001, "head_lr": 0.001, "weight_decay": 0.01, "warmup": 0.1, "grad_clip": 1.0, "dropout": 0.1, "loss": "ce", "o_weight": 1.0, "fp16": true, "refit_all": false, "rank": 16, "alpha": 32, "targets": ["qkv", "proj", "w12", "w3"]}
  params total 112.60M | trainable 3.067M (2.72%) | layers 12 | causal


/tmp/ipykernel_49/2279519611.py:122: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


  ep  1/20  loss 0.7067  val macro-F1 0.4358 (P 0.467 R 0.465)  *  [33s]
  ep  2/20  loss 0.1019  val macro-F1 0.3794 (P 0.467 R 0.368)  [29s]
  ep  3/20  loss 0.0853  val macro-F1 0.3971 (P 0.396 R 0.445)  [31s]
  ep  4/20  loss 0.0635  val macro-F1 0.4050 (P 0.415 R 0.435)  [30s]
  ep  5/20  loss 0.0546  val macro-F1 0.4593 (P 0.467 R 0.488)  *  [30s]
  ep  6/20  loss 0.0443  val macro-F1 0.4389 (P 0.450 R 0.469)  [30s]
  ep  7/20  loss 0.0343  val macro-F1 0.4279 (P 0.423 R 0.446)  [30s]
  ep  8/20  loss 0.0232  val macro-F1 0.4584 (P 0.472 R 0.477)  [30s]
  ep  9/20  loss 0.0157  val macro-F1 0.4608 (P 0.456 R 0.504)  *  [30s]
  ep 10/20  loss 0.0119  val macro-F1 0.4630 (P 0.468 R 0.475)  *  [30s]
  ep 11/20  loss 0.0072  val macro-F1 0.4644 (P 0.454 R 0.491)  *  [30s]
  ep 12/20  loss 0.0053  val macro-F1 0.4811 (P 0.478 R 0.504)  *  [30s]
  ep 13/20  loss 0.0039  val macro-F1 0.4945 (P 0.487 R 0.511)  *  [29s]
  ep 14/20  loss 0.0029  val macro-F1 0.4789 (P 0.487 R 0.480)  [29s]

In [11]:
# ---- patch: header context + error report (paste into a new cell, run after sections 1-8) ----
DEFAULTS.setdefault("header", 0)

def collate(items, with_labels=True, header=0):
    """Returns ids, key mask, labels and, per item, the offset where the window's own tokens start.
    With header > 0, windows that do not start the document get [first `header` doc tokens] + [EOS] in front;
    those prefix tokens are context only (label IGNORE, never decoded)."""
    pres = [np.concatenate([d.ids[:header], [PAD_ID]]) if (header and s > 0) else np.zeros(0, dtype=np.int64)
            for d, s, e in items]
    L = max(len(p) + e - s for p, (_, s, e) in zip(pres, items))
    ids = torch.full((len(items), L), PAD_ID, dtype=torch.long)
    km = torch.zeros((len(items), L), dtype=torch.bool)
    lab = torch.full((len(items), L), IGNORE, dtype=torch.long)
    offs = []
    for i, (p, (d, s, e)) in enumerate(zip(pres, items)):
        o = len(p)
        if o:
            ids[i, :o] = torch.from_numpy(p.astype(np.int64))
        ids[i, o:o + e - s] = torch.from_numpy(d.ids[s:e]); km[i, :o + e - s] = True
        if with_labels and d.labels is not None:
            lab[i, o:o + e - s] = torch.from_numpy(d.labels[s:e])
        offs.append(o)
    return ids, km, lab, offs

@torch.no_grad()
def predict_docs(model, docs, hp, device=DEVICE, return_probs=False):
    """Sliding windows -> centre-weighted average of probabilities -> constrained Viterbi -> spans."""
    model.eval()
    amp = torch.autocast("cuda", dtype=torch.float16) if (hp["fp16"] and device.type == "cuda") else contextlib.nullcontext()
    items = make_windows(docs, hp["window"], hp["stride"])
    acc = {d.id: np.zeros((d.n, N_TAGS), dtype=np.float32) for d in docs}
    wsum = {d.id: np.zeros(d.n, dtype=np.float32) for d in docs}
    for k in range(0, len(items), hp["eval_batch_size"]):
        batch = items[k:k + hp["eval_batch_size"]]
        ids, km, _, offs = collate(batch, with_labels=False, header=hp["header"])
        with amp:
            logits = model(ids.to(device), km.to(device))
        probs = torch.softmax(logits.float(), -1).cpu().numpy()
        for i, (d, s, e) in enumerate(batch):
            pos = np.arange(s, e)
            if hp["bidirectional"]:
                left = pos - s if s > 0 else np.full(e - s, 10 ** 6)
                right = (e - 1) - pos if e < d.n else np.full(e - s, 10 ** 6)
                w = np.minimum(np.minimum(left, right), hp["window"]) + 1.0
            else:
                w = (pos - s) + 1.0 if s > 0 else np.full(e - s, float(hp["window"]))   # more left context = better
            acc[d.id][s:e] += probs[i, offs[i]:offs[i] + e - s] * w[:, None]
            wsum[d.id][s:e] += w
    out, probs_out = {}, {}
    for d in docs:
        p = acc[d.id][d.real] / np.maximum(wsum[d.id][d.real], 1e-9)[:, None]
        out[d.id] = tags_to_spans(d, viterbi(np.log(p + 1e-9)))
        if return_probs:
            probs_out[d.id] = p
    return (out, probs_out) if return_probs else out

def train_model(model, hp, method, train_docs, val_docs, epochs, out_dir, select=True, teacher=None):
    model = model.to(DEVICE)
    head_params = list(model.head.parameters())
    head_ids = {id(p) for p in head_params}
    body = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    decay = [p for p in body if p.ndim >= 2]
    no_decay = [p for p in body if p.ndim < 2]
    opt = torch.optim.AdamW([{"params": decay, "lr": hp["lr"], "weight_decay": hp["weight_decay"]},
                             {"params": no_decay, "lr": hp["lr"], "weight_decay": 0.0},
                             {"params": head_params, "lr": hp["head_lr"], "weight_decay": 0.0}])
    items = make_windows(train_docs, hp["window"], hp["stride"])
    steps_per_epoch = math.ceil(len(items) / hp["batch_size"])
    total = steps_per_epoch * epochs
    warm = max(1, int(hp["warmup"] * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * max(0.0, (total - s) / max(1, total - warm)))
    use_amp = hp["fp16"] and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    cw = class_weights(train_docs, hp).to(DEVICE)
    rng = random.Random(SEED)
    history, best, bad = [], None, 0
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)
    t_train = 0.0
    for ep in range(1, epochs + 1):
        model.train()
        rng.shuffle(items)
        t0 = time.time()
        tot_loss, nb = 0.0, 0
        for k in range(0, len(items), hp["batch_size"]):
            ids, km, lab, _ = collate(items[k:k + hp["batch_size"]], header=hp["header"])
            ids, km, lab = ids.to(DEVICE), km.to(DEVICE), lab.to(DEVICE)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(ids, km)
            logits = logits.float()
            valid = lab != IGNORE
            loss = F.cross_entropy(logits[valid], lab[valid], weight=cw)
            if teacher is not None:
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=TEACHER_DEVICE.type == "cuda"):
                    t_logits = teacher(ids.to(TEACHER_DEVICE), km.to(TEACHER_DEVICE))
                t_logits = t_logits.float().to(DEVICE)[valid]
                T = hp["kd_T"]
                kd = F.kl_div(F.log_softmax(logits[valid] / T, -1), F.softmax(t_logits / T, -1), reduction="batchmean") * T * T
                loss = (1 - hp["kd_alpha"]) * loss + hp["kd_alpha"] * kd
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], hp["grad_clip"])
            scaler.step(opt); scaler.update(); sched.step()
            tot_loss += loss.item(); nb += 1
        t_train += time.time() - t0
        rec = {"epoch": ep, "train_loss": tot_loss / max(nb, 1), "epoch_time_s": time.time() - t0}
        if select:
            t1 = time.time()
            preds = predict_docs(model, val_docs, hp)
            sc = strict_scores({d.id: d.gold for d in val_docs}, preds)
            rec.update({"val_macro_f1": sc["macro_f1"], "val_macro_p": sc["macro_p"], "val_macro_r": sc["macro_r"],
                        "val_time_s": time.time() - t1})
            improved = best is None or sc["macro_f1"] > best["val_macro_f1"]
            if improved:
                best = {**rec, "per_label": sc["per_label"], "preds": preds}
                torch.save(model.state_dict(), os.path.join(out_dir, "best_model.pt"))
                bad = 0
            else:
                bad += 1
            print(f"  ep {ep:>2}/{epochs}  loss {rec['train_loss']:.4f}  val macro-F1 {sc['macro_f1']:.4f} "
                  f"(P {sc['macro_p']:.3f} R {sc['macro_r']:.3f}){'  *' if improved else ''}  [{rec['epoch_time_s']:.0f}s]", flush=True)
            history.append(rec)
            if bad >= hp["patience"]:
                print(f"  early stop: no improvement for {hp['patience']} epochs")
                break
        else:
            print(f"  ep {ep:>2}/{epochs}  loss {rec['train_loss']:.4f}  [{rec['epoch_time_s']:.0f}s]", flush=True)
            history.append(rec)
    if not select:
        torch.save(model.state_dict(), os.path.join(out_dir, "best_model.pt"))
    peak = torch.cuda.max_memory_allocated(DEVICE) / 2 ** 30 if DEVICE.type == "cuda" else None
    return history, best, t_train, peak

def run_experiment(name, method, **overrides):
    assert method in METHOD_DEFAULTS, method
    hp = {**DEFAULTS, **METHOD_DEFAULTS[method], **overrides}
    out_dir = os.path.join(OUT_ROOT, name)
    os.makedirs(out_dir, exist_ok=True)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    print("=" * 100 + f"\n{name}  |  method={method}\n  hp={json.dumps({k: (list(v) if isinstance(v, tuple) else v) for k, v in hp.items()})}")
    model, cfg = build_model(method, hp)
    total, trainable = count_params(model)
    print(f"  params total {total / 1e6:.2f}M | trainable {trainable / 1e6:.3f}M ({100 * trainable / total:.2f}%) | "
          f"layers {cfg['n_layer']} | {'bidirectional' if hp['bidirectional'] else 'causal'}")
    json.dump({"method": method, "hp": hp, "model_cfg": cfg}, open(os.path.join(out_dir, "run_config.json"), "w"), indent=1)
    need = hp["window"] + (hp["header"] + 1 if hp["header"] else 0)
    assert need <= cfg["max_seq_len"], f"window + header = {need} exceeds the model context {cfg['max_seq_len']}"
    teacher = None
    if method == "kd":
        teacher, thp = load_teacher(hp["teacher"])
        for k in ("window", "bidirectional", "header"):
            assert thp.get(k, 0) == hp[k], f"teacher/student setting '{k}' must match ({thp.get(k)} vs {hp[k]})"

    train_docs = [DOCS[d] for d in TRAIN_IDS]
    val_docs = [DOCS[d] for d in VAL_IDS]
    history, best, t_train, peak = train_model(model, hp, method, train_docs, val_docs, hp["epochs"], out_dir, teacher=teacher)
    off_f1, off_txt = official_eval({d.id: d.gold for d in val_docs}, best["preds"], name)
    print(f"  BEST epoch {best['epoch']}: val strict macro-F1 {best['val_macro_f1']:.4f} (official evaluator {off_f1:.4f})")

    # fold3 predictions with the selected checkpoint
    model.load_state_dict(torch.load(os.path.join(out_dir, "best_model.pt"), map_location=DEVICE))
    t0 = time.time()
    test_preds = predict_docs(model, [TEST_DOCS[d] for d in TEST_IDS], hp)
    infer_s = time.time() - t0
    write_submission(test_preds, os.path.join(out_dir, f"submission_{name}.zip"))

    refit = None
    if hp["refit_all"]:
        print(f"  refit on all {len(TRAIN_IDS) + len(VAL_IDS)} docs for {best['epoch']} epochs")
        random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
        model2, _ = build_model(method, hp)
        rdir = os.path.join(out_dir, "refit"); os.makedirs(rdir, exist_ok=True)
        train_model(model2, hp, method, train_docs + val_docs, [], best["epoch"], rdir, select=False, teacher=teacher)
        write_submission(predict_docs(model2, [TEST_DOCS[d] for d in TEST_IDS], hp), os.path.join(out_dir, f"submission_{name}_refit.zip"))
        refit = True
        del model2

    metrics = {"name": name, "method": method, "params_total": total, "params_trainable": trainable,
               "layers": cfg["n_layer"], "bidirectional": hp["bidirectional"],
               "best_epoch": best["epoch"], "val_macro_f1": best["val_macro_f1"], "val_macro_f1_official": off_f1,
               "val_macro_p": best["val_macro_p"], "val_macro_r": best["val_macro_r"],
               "per_label_f1": {l: v["f1"] for l, v in best["per_label"].items()},
               "per_label": best["per_label"], "train_time_s": t_train, "peak_gpu_gb": peak,
               "test_infer_s": infer_s, "test_infer_ms_per_doc": 1000 * infer_s / max(1, len(TEST_IDS)),
               "checkpoint_mb": os.path.getsize(os.path.join(out_dir, "best_model.pt")) / 2 ** 20,
               "history": history, "hp": hp, "refit": refit}
    json.dump(metrics, open(os.path.join(out_dir, "metrics.json"), "w"), indent=1)
    json.dump(best["preds"], open(os.path.join(out_dir, "val_predictions.json"), "w"))
    print(f"  wrote {out_dir}/submission_{name}.zip and metrics.json")
    del model, teacher
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return metrics

def error_report(name, n_examples=4):
    """Validation error breakdown for one run: per label TP / boundary errors / wrong label / missed / spurious,
    the most frequent label confusions, and examples with context."""
    preds = json.load(open(os.path.join(OUT_ROOT, name, "val_predictions.json")))
    rows = {l: Counter() for l in LABELS}
    conf, ex = Counter(), defaultdict(list)
    for did in VAL_IDS:
        T = ALL_TEXT[did]
        G = set(map(tuple, DOCS[did].gold)); P = set(tuple(x[:3]) for x in preds.get(did, []))
        used = set()
        for g in G & P:
            rows[g[2]]["tp"] += 1
        for p in sorted(P - G):
            ov = [g for g in G - P if min(p[1], g[1]) > max(p[0], g[0])]
            same = [g for g in ov if g[2] == p[2]]
            if same:
                rows[p[2]]["boundary"] += 1; used.add(same[0])
                ex[("boundary", p[2])].append(f"gold [{T[same[0][0]:same[0][1]]}]  pred [{T[p[0]:p[1]]}]")
            elif ov:
                rows[ov[0][2]]["wrong_label"] += 1; used.add(ov[0]); conf[f"{ov[0][2]} -> {p[2]}"] += 1
                ex[("label", ov[0][2])].append(f"[{T[p[0]:p[1]]}] gold {ov[0][2]} pred {p[2]}")
            else:
                rows[p[2]]["spurious"] += 1
                ex[("spurious", p[2])].append(f"...{T[max(0, p[0] - 40):p[0]]}[{T[p[0]:p[1]]}]{T[p[1]:p[1] + 30]}...")
        for g in sorted(G - P):
            if g not in used:
                rows[g[2]]["missed"] += 1
                ex[("missed", g[2])].append(f"...{T[max(0, g[0] - 40):g[0]]}[{T[g[0]:g[1]]}]{T[g[1]:g[1] + 30]}...")
    print(f"{'label':<11}{'gold':>6}{'TP':>6}{'boundary':>10}{'wrong lab':>11}{'missed':>8}{'spurious':>10}")
    for l in LABELS:
        r = rows[l]
        print(f"{l:<11}{r['tp'] + r['boundary'] + r['wrong_label'] + r['missed']:>6}{r['tp']:>6}{r['boundary']:>10}"
              f"{r['wrong_label']:>11}{r['missed']:>8}{r['spurious']:>10}")
    print("\nmost frequent label confusions (gold -> predicted):", dict(conf.most_common(10)))
    for l in LABELS:
        for kind in ("boundary", "label", "missed", "spurious"):
            if ex[(kind, l)]:
                print(f"\n{l} / {kind}:")
                for e_ in ex[(kind, l)][:n_examples]:
                    print("   ", e_.replace("\n", " "))

In [12]:
results = {}
results["lorma_h256"] = run_experiment("lorma_h256", "lorma", epochs=40, patience=10, header=256)
results["lorma_h256_wce"] = run_experiment("lorma_h256_wce", "lorma", epochs=40, patience=10, header=256, loss="weighted")
summary()
error_report("lorma_h256")

lorma_h256  |  method=lorma
  hp={"bidirectional": true, "window": 512, "stride": 384, "epochs": 40, "patience": 10, "batch_size": 8, "eval_batch_size": 32, "lr": 0.001, "head_lr": 0.001, "weight_decay": 0.01, "warmup": 0.1, "grad_clip": 1.0, "dropout": 0.1, "loss": "ce", "o_weight": 1.0, "fp16": true, "refit_all": false, "header": 256, "rank": 16, "alpha": 32, "targets": ["qkv", "proj", "w12", "w3"]}
  params total 112.60M | trainable 3.067M (2.72%) | layers 12 | bidirectional


/tmp/ipykernel_49/3441274576.py:104: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


  ep  1/40  loss 0.9223  val macro-F1 0.2626 (P 0.263 R 0.264)  *  [53s]
  ep  2/40  loss 0.1130  val macro-F1 0.3625 (P 0.451 R 0.340)  *  [51s]
  ep  3/40  loss 0.0871  val macro-F1 0.2835 (P 0.334 R 0.272)  [50s]
  ep  4/40  loss 0.0808  val macro-F1 0.3342 (P 0.404 R 0.355)  [50s]
  ep  5/40  loss 0.0837  val macro-F1 0.3689 (P 0.392 R 0.374)  *  [50s]
  ep  6/40  loss 0.0696  val macro-F1 0.4702 (P 0.486 R 0.491)  *  [50s]
  ep  7/40  loss 0.0588  val macro-F1 0.4607 (P 0.507 R 0.455)  [50s]
  ep  8/40  loss 0.0571  val macro-F1 0.4490 (P 0.515 R 0.444)  [50s]
  ep  9/40  loss 0.0499  val macro-F1 0.4679 (P 0.520 R 0.464)  [50s]
  ep 10/40  loss 0.0428  val macro-F1 0.4742 (P 0.556 R 0.477)  *  [50s]
  ep 11/40  loss 0.0387  val macro-F1 0.4227 (P 0.484 R 0.402)  [50s]
  ep 12/40  loss 0.0366  val macro-F1 0.4628 (P 0.502 R 0.498)  [50s]
  ep 13/40  loss 0.0314  val macro-F1 0.4089 (P 0.463 R 0.426)  [50s]
  ep 14/40  loss 0.0274  val macro-F1 0.4299 (P 0.464 R 0.440)  [50s]
  ep 

In [1]:
import os, shutil, zipfile
from IPython.display import FileLink, display

runs = ["lorma_h256", "lorma_h256_wce"]
os.chdir("/kaggle/working")                       # FileLink only works for files under /kaggle/working

for r in runs:
    src = f"/kaggle/working/lner/{r}/submission_{r}.zip"
    if os.path.exists(src):
        shutil.copy(src, f"submission_{r}.zip")   # copy to the top of /kaggle/working
        display(FileLink(f"submission_{r}.zip"))  # click to download
    else:
        print("not found (run not finished?):", src)

# optional: one zip with both submissions + their metrics, for your records
with zipfile.ZipFile("tuning_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for r in runs:
        for f in [f"submission_{r}.zip", "metrics.json", "val_predictions.json"]:
            p = f"/kaggle/working/lner/{r}/{f}"
            if os.path.exists(p):
                z.write(p, arcname=f"{r}/{f}")
display(FileLink("tuning_results.zip"))

/kaggle/working/submission_lorma_h256.zip

/kaggle/working/submission_lorma_h256_wce.zip

/kaggle/working/tuning_results.zip